# SBERT NLI Training — Paper Replication

This notebook implements the **core Sentence-BERT NLI training experiment** from Reimers & Gurevych (2019).

### Target experiment

```text
Pretrained BERT
      ↓
Mean Pooling
      ↓
Siamese / tied-weight encoder
      ↓
[u ; v ; |u-v|]
      ↓
3-class NLI classifier
      ↓
SNLI + MultiNLI
```

The paper trains SBERT for **one epoch** on SNLI + MultiNLI using a 3-way classification objective, batch size 16, Adam with learning rate `2e-5`, and 10% linear warm-up. Mean pooling is the default pooling strategy.

Our first target is the paper's **SBERT-NLI-base** result from Table 1. After training, the classifier head will be discarded and the BERT + mean-pooling encoder will be evaluated as a sentence encoder on STS12–STS16, STS-B, and SICK-R.

**Implementation note:** the paper text does not specify a maximum sequence length or gradient accumulation. Because this project is being run on a 6 GB GPU, we use dynamic padding, a small physical batch, and gradient accumulation to obtain an effective batch size of 16. These are hardware-adaptation choices, not claims about the original hardware configuration.


In [20]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU memory: 6.0 GB


In [21]:
# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False

print("Seed:", SEED)


Seed: 42


In [22]:
# ============================================================
# 3. CONFIGURATION
# ============================================================

MODEL_NAME = "bert-base-uncased"

DATA_DIR = Path("..") / "data"
NLI_PATH = DATA_DIR / "NLI" / "NLI"

OUTPUT_DIR = Path("outputs") / "sbert_nli_base"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Paper configuration
EPOCHS = 1
LEARNING_RATE = 2e-5
PAPER_BATCH_SIZE = 16
WARMUP_RATIO = 0.10

# Hardware-adapted implementation:
# 4 physical examples x 4 accumulation steps = effective batch size 16.
PHYSICAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = PAPER_BATCH_SIZE // PHYSICAL_BATCH_SIZE

# Practical value for the 6 GB GPU.
# The paper text does not specify max sequence length.
MAX_LENGTH = 128

NUM_LABELS = 3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Physical batch size:", PHYSICAL_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", PHYSICAL_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", EPOCHS)
print("Warmup ratio:", WARMUP_RATIO)
print("Max length:", MAX_LENGTH)


Model: bert-base-uncased
Device: cuda
Physical batch size: 4
Gradient accumulation: 4
Effective batch size: 16
Learning rate: 2e-05
Epochs: 1
Warmup ratio: 0.1
Max length: 128


In [23]:
# ============================================================
# 4. LOAD THE COMBINED NLI DATASET
# ============================================================

if not NLI_PATH.exists():
    raise FileNotFoundError(
        f"Combined NLI dataset not found at: {NLI_PATH.resolve()}\n"
        "Run data.py first."
    )

nli_dataset = load_from_disk(str(NLI_PATH))

print(nli_dataset)
print()
print("Number of examples:", f"{len(nli_dataset):,}")
print("Columns:", nli_dataset.column_names)


Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 942069
})

Number of examples: 942,069
Columns: ['premise', 'hypothesis', 'label']


In [24]:
# ============================================================
# 5. FINAL NLI SANITY CHECK
# ============================================================

labels = np.asarray(nli_dataset["label"])

print("Unique labels:", sorted(np.unique(labels).tolist()))
print("Label counts:")
print(pd.Series(labels).value_counts().sort_index())

assert set(np.unique(labels)).issubset({0, 1, 2}),     "Invalid NLI labels found."

assert len(nli_dataset) > 0,     "NLI dataset is empty."

print()
print("NLI dataset is valid for 3-class training.")


Unique labels: [0, 1, 2]
Label counts:
0    314315
1    313664
2    314090
Name: count, dtype: int64

NLI dataset is valid for 3-class training.


In [25]:
# ============================================================
# 6. TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer:", MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)
print("PAD token:", tokenizer.pad_token)
print("CLS token:", tokenizer.cls_token)
print("SEP token:", tokenizer.sep_token)


Tokenizer: bert-base-uncased
Vocabulary size: 30522
PAD token: [PAD]
CLS token: [CLS]
SEP token: [SEP]


In [26]:
# ============================================================
# 7. COLLATE FUNCTION
# ============================================================

def collate_fn(batch):
    """Tokenize a batch of NLI sentence pairs with dynamic padding."""

    premises = [example["premise"] for example in batch]
    hypotheses = [example["hypothesis"] for example in batch]
    labels = torch.tensor(
        [example["label"] for example in batch],
        dtype=torch.long
    )

    premise_inputs = tokenizer(
        premises,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    hypothesis_inputs = tokenizer(
        hypotheses,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    return {
        "premise": premise_inputs,
        "hypothesis": hypothesis_inputs,
        "labels": labels,
    }


train_loader = DataLoader(
    nli_dataset,
    batch_size=PHYSICAL_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("Number of physical batches:", len(train_loader))


Number of physical batches: 235518


## Mean pooling

BERT produces one contextual vector for every token.

If:

\[
H = [h_1,h_2,\ldots,h_T]
\]

and `mask` indicates which positions are real tokens rather than padding, mean pooling computes:

\[
u =

rac{\sum_{i=1}^{T} mask_i h_i}
     {\sum_{i=1}^{T} mask_i}
\]

This gives one fixed-size sentence vector.

The paper uses **MEAN pooling as its default configuration**.


In [27]:
# ============================================================
# 8. MEAN POOLING
# ============================================================

def mean_pooling(last_hidden_state, attention_mask):
    """Mean-pool token representations while ignoring padding."""

    mask = attention_mask.unsqueeze(-1).expand(
        last_hidden_state.size()
    ).float()

    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)

    return summed / counts


## SBERT NLI architecture

For each sentence pair:

\[
u = MeanPool(BERT(A))
\]

\[
v = MeanPool(BERT(B))
\]

Then construct:

\[
x = [u;v;|u-v|]
\]

For BERT-base:

\[
u,v \in \mathbb{R}^{768}
\]

so:

\[
x \in \mathbb{R}^{2304}
\]

The classification layer maps:

```text
2304 → 3
```

The two sentence encoders use the **same BERT object**, so their weights are tied.


In [28]:
# ============================================================
# 9. SBERT NLI MODEL
# ============================================================

class SBERTForNLI(nn.Module):

    def __init__(self, model_name, num_labels=3):
        super().__init__()

        # One BERT encoder is shared by both sentences.
        self.bert = AutoModel.from_pretrained(model_name)

        self.hidden_size = self.bert.config.hidden_size

        self.classifier = nn.Linear(
            self.hidden_size * 3,
            num_labels
        )

    def encode(self, input_ids, attention_mask, token_type_ids=None):

        kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }

        # BERT uses token_type_ids; this keeps the encoder interface
        # compatible with models that may not use them.
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        outputs = self.bert(**kwargs)

        embedding = mean_pooling(
            outputs.last_hidden_state,
            attention_mask
        )

        return embedding

    def forward(self, premise, hypothesis):

        u = self.encode(
            input_ids=premise["input_ids"],
            attention_mask=premise["attention_mask"],
            token_type_ids=premise.get("token_type_ids"),
        )

        v = self.encode(
            input_ids=hypothesis["input_ids"],
            attention_mask=hypothesis["attention_mask"],
            token_type_ids=hypothesis.get("token_type_ids"),
        )

        difference = torch.abs(u - v)

        combined = torch.cat(
            [u, v, difference],
            dim=1
        )

        logits = self.classifier(combined)

        return logits, u, v


model = SBERTForNLI(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS
)

model.to(DEVICE)

print(model.classifier)
print()
print("BERT hidden size:", model.hidden_size)
print("Classifier input size:", model.classifier.in_features)


Linear(in_features=2304, out_features=3, bias=True)

BERT hidden size: 768
Classifier input size: 2304


In [29]:
# ============================================================
# 10. VERIFY ARCHITECTURE
# ============================================================

bert_parameter_count = sum(
    p.numel() for p in model.bert.parameters()
)

total_parameter_count = sum(
    p.numel() for p in model.parameters()
)

print(f"BERT parameters:  {bert_parameter_count:,}")
print(f"Total parameters: {total_parameter_count:,}")

assert model.classifier.in_features == 2304

print()
print("Architecture sanity check passed.")


BERT parameters:  109,482,240
Total parameters: 109,489,155

Architecture sanity check passed.


In [30]:
# ============================================================
# 11. TEST ONE FORWARD PASS
# ============================================================

batch = next(iter(train_loader))

batch_premise = {
    key: value.to(DEVICE)
    for key, value in batch["premise"].items()
}

batch_hypothesis = {
    key: value.to(DEVICE)
    for key, value in batch["hypothesis"].items()
}

batch_labels = batch["labels"].to(DEVICE)

model.eval()

with torch.no_grad():
    logits, u, v = model(
        batch_premise,
        batch_hypothesis
    )

print("Premise embedding shape:", tuple(u.shape))
print("Hypothesis embedding shape:", tuple(v.shape))
print("Logits shape:", tuple(logits.shape))
print("Labels shape:", tuple(batch_labels.shape))

assert u.shape[1] == 768
assert v.shape[1] == 768
assert logits.shape[1] == 3

print()
print("Forward pass passed.")


Premise embedding shape: (4, 768)
Hypothesis embedding shape: (4, 768)
Logits shape: (4, 3)
Labels shape: (4,)

Forward pass passed.


In [31]:
# ============================================================
# 12. LOSS, OPTIMIZER, AND SCHEDULER
# ============================================================

criterion = nn.CrossEntropyLoss()

# The paper specifies Adam with learning rate 2e-5.
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

steps_per_epoch = math.ceil(
    len(train_loader) / GRADIENT_ACCUMULATION_STEPS
)

total_optimizer_steps = steps_per_epoch * EPOCHS

warmup_steps = int(
    total_optimizer_steps * WARMUP_RATIO
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimizer_steps,
)

print("Physical batches:", len(train_loader))
print("Optimizer steps:", total_optimizer_steps)
print("Warmup steps:", warmup_steps)


Physical batches: 235518
Optimizer steps: 58880
Warmup steps: 5888


In [32]:
# ============================================================
# 13. TRAINING LOOP
# ============================================================

def train_one_epoch(
    model,
    dataloader,
    optimizer,
    scheduler,
    criterion,
    device,
    accumulation_steps,
    use_amp=True,
):
    model.train()

    running_loss = 0.0

    optimizer.zero_grad(set_to_none=True)

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp
    )

    progress = tqdm(
        dataloader,
        desc="Training",
        leave=True
    )

    for step, batch in enumerate(progress):

        # ----------------------------------------------------
        # Move data to device
        # ----------------------------------------------------

        premise = {
            key: value.to(
                device,
                non_blocking=True
            )
            for key, value in batch["premise"].items()
        }

        hypothesis = {
            key: value.to(
                device,
                non_blocking=True
            )
            for key, value in batch["hypothesis"].items()
        }

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        with torch.amp.autocast(
            device_type="cuda",
            enabled=use_amp
        ):

            logits, _, _ = model(
                premise,
                hypothesis
            )

            loss = criterion(
                logits,
                labels
            )

            # Gradient accumulation:
            # physical batch 4 × 4 accumulation = effective 16
            loss_for_backward = (
                loss / accumulation_steps
            )

        # ----------------------------------------------------
        # Backward pass
        # ----------------------------------------------------

        scaler.scale(
            loss_for_backward
        ).backward()

        running_loss += loss.item()

        # ----------------------------------------------------
        # Optimizer update
        # ----------------------------------------------------

        is_last_batch = (
            (step + 1) == len(dataloader)
        )

        if (
            (step + 1) % accumulation_steps == 0
            or is_last_batch
        ):

            scaler.step(optimizer)

            scaler.update()

            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )

        # ----------------------------------------------------
        # Progress display
        # ----------------------------------------------------

        if (step + 1) % 100 == 0:

            average_loss = (
                running_loss / (step + 1)
            )

            progress.set_postfix(
                loss=f"{average_loss:.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}"
            )

    return running_loss / len(dataloader)

## Training

This is the main paper replication step.

The paper trains SBERT for **one epoch** on SNLI + MultiNLI using:

- 3-way NLI classification
- batch size 16
- Adam
- learning rate `2e-5`
- linear warm-up over 10% of the training data
- MEAN pooling

Our physical batch is smaller because of GPU memory, but gradient accumulation gives an effective batch size of 16.


In [33]:
# ============================================================
# 14. TRAIN SBERT FOR ONE EPOCH
# ============================================================

training_history = []

for epoch in range(EPOCHS):

    print()
    print("=" * 70)
    print(f"EPOCH {epoch + 1}/{EPOCHS}")
    print("=" * 70)

    avg_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=DEVICE,
        accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        use_amp=USE_AMP,
    )

    training_history.append({
        "epoch": epoch + 1,
        "loss": avg_loss,
        "learning_rate": scheduler.get_last_lr()[0],
    })

    print(f"Average training loss: {avg_loss:.4f}")



EPOCH 1/1


Training: 100%|██████████| 235518/235518 [8:38:46<00:00,  7.57it/s, loss=0.5941, lr=1.89e-09]        

Average training loss: 0.5941


In [34]:
# ============================================================
# 15. TRAINING SUMMARY
# ============================================================

history_df = pd.DataFrame(training_history)

display(history_df)

print()
print("Training completed.")


,epoch,loss,learning_rate
0,1,0.594119,0.0



Training completed.


In [35]:
# ============================================================
# 16. SAVE THE SBERT MODEL
# ============================================================

MODEL_OUTPUT = OUTPUT_DIR / "model"
TOKENIZER_OUTPUT = OUTPUT_DIR / "tokenizer"

MODEL_OUTPUT.mkdir(parents=True, exist_ok=True)
TOKENIZER_OUTPUT.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_name": MODEL_NAME,
        "pooling": "mean",
        "num_labels": NUM_LABELS,
        "max_length": MAX_LENGTH,
        "seed": SEED,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "effective_batch_size": PAPER_BATCH_SIZE,
        "physical_batch_size": PHYSICAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    },
    MODEL_OUTPUT / "sbert_nli_base.pt"
)

tokenizer.save_pretrained(str(TOKENIZER_OUTPUT))

history_df.to_csv(
    OUTPUT_DIR / "training_history.csv",
    index=False
)

print("Saved model to:", MODEL_OUTPUT.resolve())
print("Saved tokenizer to:", TOKENIZER_OUTPUT.resolve())


Saved model to: F:\SEM-7\NLP\Project\notebooks\outputs\sbert_nli_base\model
Saved tokenizer to: F:\SEM-7\NLP\Project\notebooks\outputs\sbert_nli_base\tokenizer


# Next: Table 1 Evaluation

The trained model is now ready for the actual **Table 1 replication**.

The next notebook will:

```text
trained SBERT
      ↓
discard NLI classifier
      ↓
Mean Pooling
      ↓
sentence embeddings
      ↓
cosine similarity
      ↓
STS12
STS13
STS14
STS15
STS16
STS-B
SICK-R
      ↓
Spearman correlation
      ↓
compare with paper Table 1
```

Paper target for `SBERT-NLI-base`:

| Dataset | Paper |
|---|---:|
| STS12 | 70.97 |
| STS13 | 76.53 |
| STS14 | 73.19 |
| STS15 | 79.09 |
| STS16 | 74.30 |
| STS-B | 77.03 |
| SICK-R | 72.91 |
| **Average** | **74.89** |


In [36]:
from pathlib import Path
from datasets import load_from_disk

NLI_PATH = Path("..") / "data" / "NLI" / "NLI"

print("Path:")
print(NLI_PATH.resolve())

print("\nExists:", NLI_PATH.exists())

if NLI_PATH.exists():
    print("\nContents:")
    for item in NLI_PATH.iterdir():
        print(" ", item.name)

    nli_dataset = load_from_disk(str(NLI_PATH))

    print("\nDataset:")
    print(nli_dataset)

    print("\nNumber of examples:")
    print(f"{len(nli_dataset):,}")

    print("\nColumns:")
    print(nli_dataset.column_names)

Path:
F:\SEM-7\NLP\Project\data\NLI\NLI

Exists: True

Contents:
  data-00000-of-00001.arrow
  dataset_info.json
  state.json

Dataset:
Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 942069
})

Number of examples:
942,069

Columns:
['premise', 'hypothesis', 'label']
